In [ ]:
import numpy as np
import pandas as pd
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
train_df = pd.read_csv("data/raw/fraudTrain.csv")

In [3]:
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])

In [5]:

def add_basic_features(df):
    df = df.copy()
    ts = df["trans_date_trans_time"]
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 3)).astype(int)
    df["log_amt"] = np.log1p(df["amt"])
    df["age"] = (ts - pd.to_datetime(df["dob"])).dt.days / 365.25

    # haversine distance (km) between customer home and merchant
    lat1, lon1 = np.radians(df["lat"]), np.radians(df["long"])
    lat2, lon2 = np.radians(df["merch_lat"]), np.radians(df["merch_long"])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))
    return df



In [6]:
train_fe = add_basic_features(train_df)
train_fe[["hour", "is_night", "log_amt", "age", "distance_km"]].describe().round(2)

/var/folders/hp/sbkd0ccx7h3_3210cd7jj9700000gn/T/ipykernel_8176/3520143131.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = ts.dt.dayofweek


,hour,is_night,log_amt,age,distance_km
count,1296675.00,1296675.00,1296675.00,1296675.00,1296675.00
mean,12.80,0.24,3.53,46.00,76.11
std,6.82,0.42,1.29,17.39,29.12
min,0.00,0.00,0.69,13.92,0.02
25%,7.00,0.00,2.37,32.60,55.33
50%,14.00,0.00,3.88,43.97,78.23
75%,19.00,0.00,4.43,57.07,98.50
max,23.00,1.00,10.27,95.64,152.12


In [7]:
train_fe.groupby("is_fraud").distance_km.describe().round(1)

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,76.1,29.1,0.0,55.3,78.2,98.5,152.1
1,7506.0,76.3,28.8,0.7,55.6,77.9,98.4,144.5


In [8]:
def add_card_history_features(df):
    df = df.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)
    g = df.groupby("cc_num")

    n = g.cumcount()                                   # transactions before this one
    s1 = g["amt"].cumsum() - df["amt"]                 # sum of PAST amounts only
    s2 = (df["amt"] ** 2).groupby(df["cc_num"]).cumsum() - df["amt"] ** 2

    mean_prev = s1 / n.replace(0, np.nan)
    std_prev = np.sqrt((s2 / n.replace(0, np.nan) - mean_prev ** 2).clip(lower=0))

    df["card_txn_count"] = n
    df["amt_ratio"] = (df["amt"] / mean_prev).fillna(1.0)
    df["amt_zscore"] = ((df["amt"] - mean_prev) / std_prev.replace(0, np.nan)).fillna(0)
    df["secs_since_last"] = g["trans_date_trans_time"].diff().dt.total_seconds()
    return df


In [9]:

train_fe = add_card_history_features(add_basic_features(train_df))
train_fe.groupby("is_fraud")[["amt_ratio", "amt_zscore", "secs_since_last"]].median().round(2)

/var/folders/hp/sbkd0ccx7h3_3210cd7jj9700000gn/T/ipykernel_8176/3520143131.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = ts.dt.dayofweek


,amt_ratio,amt_zscore,secs_since_last
is_fraud,,,
0,0.66,-0.19,16623.0
1,5.16,2.47,4908.0


In [10]:
from src.finguard.ml.features import build_features
train_fe = build_features(train_df)
train_fe.groupby("is_fraud")[["amt_ratio", "amt_zscore", "secs_since_last"]].median().round(2)

,amt_ratio,amt_zscore,secs_since_last
is_fraud,,,
0,0.66,-0.19,16623.0
1,5.16,2.47,4908.0
